# 03. 병합 모델 → GGUF 변환과 양자화

02에서 Drive에 저장한 병합 모델(`merged`, fp16 약 4.3GB)을
노트북 CPU에서 돌릴 수 있는 **GGUF Q4_K_M(약 1.3GB)** 로 바꿉니다.

- GPU가 필요 없는 작업이라 **런타임 유형은 CPU로 둬도 됩니다** (GPU 사용량 절약)
- 전체 20~30분 (대부분 llama.cpp 빌드 시간)
- 비교용으로 조금 더 정확한 Q5_K_M도 함께 만듭니다

In [ ]:
# 1. Drive 연결과 경로
import os
from google.colab import drive
drive.mount("/content/drive")

RUN_NAME = "kanana-lora-r16"
DRIVE_DIR = f"/content/drive/MyDrive/sejong-gpt/{RUN_NAME}"
MERGED_DIR = f"{DRIVE_DIR}/merged"
GGUF_DRIVE_DIR = f"{DRIVE_DIR}/gguf"
WORK = "/content/gguf"          # 큰 중간 파일은 Colab 디스크에서 작업 (Drive보다 빠름)
os.makedirs(GGUF_DRIVE_DIR, exist_ok=True); os.makedirs(WORK, exist_ok=True)

MODEL_NAME = "sejong-kanana-2.1b"
print(os.listdir(MERGED_DIR))   # config.json, model-*.safetensors, tokenizer 파일들이 보여야 함

In [ ]:
# 2. llama.cpp 받기와 변환용 패키지
%cd /content
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

In [ ]:
# 3. Hugging Face 형식 → GGUF (fp16, 아직 양자화 전)
F16 = f"{WORK}/{MODEL_NAME}-f16.gguf"
!python llama.cpp/convert_hf_to_gguf.py "{MERGED_DIR}" --outfile "{F16}" --outtype f16
!ls -lh {WORK}

In [ ]:
# 4. 양자화 도구 빌드 (10~15분)
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build llama.cpp/build --config Release -j 4 --target llama-quantize 2>&1 | tail -n 3

In [ ]:
# 5. 양자화: Q4_K_M (기본으로 쓸 파일), Q5_K_M (비교용)
for q in ["Q4_K_M", "Q5_K_M"]:
    out = f"{WORK}/{MODEL_NAME}-{q}.gguf"
    !llama.cpp/build/bin/llama-quantize "{F16}" "{out}" {q} 2>&1 | tail -n 2
!ls -lh {WORK}

In [ ]:
# 6. 결과를 Drive로 복사 (fp16 중간 파일은 크기만 차지하므로 제외)
import shutil, glob
for path in glob.glob(f"{WORK}/*Q*_K_M.gguf"):
    shutil.copy(path, GGUF_DRIVE_DIR)
for name in sorted(os.listdir(GGUF_DRIVE_DIR)):
    size = os.path.getsize(f"{GGUF_DRIVE_DIR}/{name}") / 1024**3
    print(f"{name}: {size:.2f} GB")

## 다음: 노트북으로 내려받기

Google Drive 웹(drive.google.com)에서 `내 드라이브/sejong-gpt/kanana-lora-r16/gguf/` 폴더를 열고
`sejong-kanana-2.1b-Q4_K_M.gguf`를 내려받아, 로컬 저장소의 `models/` 폴더에 넣으세요.
(`models/`는 `.gitignore`에 있어서 GitHub에는 올라가지 않습니다.)

`.env`의 `MODEL_PATH`도 파일 이름에 맞게 바꿉니다.

```
MODEL_PATH=models/sejong-kanana-2.1b-Q4_K_M.gguf
```